<a href="https://colab.research.google.com/github/yueop/AS_LAB/blob/main/FCN_dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import sys
from google.colab import drive

# 1. 드라이브 마운트
drive.mount('/content/drive')

# 2. 경로 설정
PROJECT_PATH = '/content/drive/MyDrive/FCN_Project'
if not os.path.exists(PROJECT_PATH):
    os.makedirs(PROJECT_PATH)

# 3. 시스템 경로 추가 및 작업 디렉토리 변경
if PROJECT_PATH not in sys.path:
    sys.path.append(PROJECT_PATH)
os.chdir(PROJECT_PATH)

print(f"현재 위치: {os.getcwd()}")

Mounted at /content/drive
현재 위치: /content/drive/MyDrive/FCN_Project


In [2]:
%%writefile dataset.py
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets
import torchvision.transforms.functional as F
import torchvision.transforms as T
from torchvision.transforms import InterpolationMode
import numpy as np
import random

class VOCSegmentationDataset(Dataset):
    def __init__(self, root_dir, image_set='train', transform=True):
        #파이토치 내장 데이터셋 활용
        self.dataset = datasets.VOCSegmentation(
            root=root_dir,
            year='2012',
            image_set=image_set,
            download=True
        )
        self.transform = transform
        self.image_set = image_set

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, index):
        #1. 이미지와 정답 마스크 불러오기
        image, mask = self.dataset[index]

        #2. 전처리(크기 조정 및 데이터 증강)
        if self.transform:
            if self.image_set == 'train': #훈련 데이터 셋일 때
                #1. 320x320 크기로 리사이징
                image = F.resize(image, (320, 320), interpolation=InterpolationMode.BILINEAR)
                mask = F.resize(mask, (320, 320), interpolation=InterpolationMode.NEAREST)

                #2. 256x256 크기로 자를 무작위 좌표(i, j, h, w) 생성
                i, j, h, w = T.RandomCrop.get_params(image, output_size=(256, 256))

                #3. 원본과 마스크를 "완전히 똑같은 좌표"로 잘라냄
                image = F.crop(image, i, j, h, w)
                mask = F.crop(mask, i, j, h, w)

                #4. 무작위 좌우 반전
                if random.random() > 0.5:
                    image = F.hflip(image)
                    mask = F.hflip(mask)

            else: #검증 데이터 셋일 때(자르거나 뒤집지 않고, 256x256 크기로 리사이징)
                image = F.resize(image, (256, 256), interpolation=InterpolationMode.BILINEAR)
                mask = F.resize(mask, (256, 256), interpolation=InterpolationMode.NEAREST)


            #3. 텐서로 변환
            image = F.to_tensor(image)

            #4. 정규화(ImageNet 공식 평균, 표준편차 사용)
            image = F.normalize(image, mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

            #마스크는 각 픽셀이 클래스 인덱스(0~20, 255)를 가지므로 Long 텐서로 변환(형태: H, W)
            mask = torch.as_tensor(np.array(mask), dtype=torch.long)

        return image, mask

def get_dataloader(data_dir, batch_size):
    #학습용 및 검증용 데이터셋
    train_dataset = VOCSegmentationDataset(data_dir, image_set='train')
    val_dataset = VOCSegmentationDataset(data_dir, image_set='val')

    #DataLoader 씌우기
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, drop_last=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

    return train_loader, val_loader

Overwriting dataset.py


마스크의 보간법(Interpolation): NEAREST

- 이미지는 사이즈를 줄일 때 픽셀을 섞어서 부드럽게(Bilinear) 만들어도 된다.
- 하지만 마스크는 픽셀값이 곧 클래스 번호이기 때문에 만약 8번 픽셀과 10번 픽셀을 섞어버리면 갑자기 9번 클래스가 생겨나는 현상이 발생한다. 따라서 정답 마스크는 무조건 가장 가까운 값을 그대로 가져오는 NEAREST 방식을 써야한다.

마스크의 텐서 타입: torch.long

- 나중에 쓰일 손실 함수(CrossEntropyLoss)는 정답 라벨이 반드시 정수형(Long) 텐서여야 작동한다.

마스크의 채널(channel) 차원이 없는 이유

- 이미지는 (C, H, W) 형태지만, 마스크는 (H, W) 형태의 2차원 배열이다. 픽셀 하나하나의 값이 정답 번호이기 때문이다.